# 4. Outlier Detection and Treatment

This notebook covers:
1. **Statistical Outlier Detection:**
   - **IQR (Interquartile Range) Method** (Best for skewed/non-normal data).
   - **Z-Score Method** (Best for approximately normal/bell-shaped data).
2. **Algorithmic Outlier Detection:**
   - **Isolation Forest** (Multi-dimensional outlier detection).
3. **Outlier Treatment Strategies:**
   - Capping / Truncation (Winsorization) vs. Dropping.
4. **Data Leakage Rule:** Computing outlier thresholds strictly on the training set.

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create a sample dataset containing realistic values + clear outliers
np.random.seed(42)
normal_salaries = np.random.normal(loc=60000, scale=15000, size=100)
outlier_salaries = [250000, 300000, 5000] # Extreme high and low points

salary_data = np.concatenate([normal_salaries, outlier_salaries])
df_outliers = pd.DataFrame({'Salary': salary_data})

print("Dataset size:", len(df_outliers))
display(df_outliers.describe())

Dataset size: 103


,Salary
count,103.000000
mean,62128.448776
std,33558.957676
min,5000.000000
25%,50982.410234
50%,58265.275764
75%,67574.861896
max,300000.000000


---
## Part 1: Statistical Method — IQR (Interquartile Range)

- **$Q_1$ (25th percentile)** and **$Q_3$ (75th percentile)**
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Bound} = Q_1 - (1.5 \times \text{IQR})$
- $\text{Upper Bound} = Q_3 + (1.5 \times \text{IQR})$

Any value outside $[\text{Lower Bound}, \text{Upper Bound}]$ is flagged as an outlier.

In [13]:
Q1 = df_outliers['Salary'].quantile(0.25)
Q3 = df_outliers['Salary'].quantile(0.75)

IQR = Q3 - Q1

lower_bound_iqr = Q1 - 1.5 * IQR 
upper_bound_iqr = Q3 + 1.5 * IQR

print(f"IQR Lower limit {lower_bound_iqr:.2f}")
print(f"IQR Upper limit {upper_bound_iqr:.2f}")

iqr_outliers = df_outliers[(df_outliers['Salary'] < lower_bound_iqr) | (df_outliers['Salary'] > upper_bound_iqr)]

print(f'outliers detected: {len(iqr_outliers)}')
display(iqr_outliers)

IQR Lower limit 26093.73
IQR Upper limit 92463.54
outliers detected: 4


,Salary,Z_Score,z_score
74,20703.823439,-1.234384,-1.234384
100,250000.000000,5.598253,5.598253
101,300000.000000,7.088169,7.088169
102,5000.000000,-1.702331,-1.702331


---
## Part 2: Statistical Method — Z-Score

- Measures how many standard deviations ($\sigma$) a data point is from the mean ($\mu$).
- Standard cutoff: $|Z| > 3$ is typically flagged as an outlier.
- **Limitation:** Highly sensitive to extreme values because mean and standard deviation themselves get distorted by outliers.

In [12]:
mean = df_outliers['Salary'].mean()
std = df_outliers['Salary'].std()

df_outliers['z_score'] = (df_outliers['Salary'] - mean) / std

z_outliers = df_outliers[df_outliers['z_score'].abs() > 3]

print(f'detected {len(z_outliers)} outliers using the z score method')
display(z_outliers)

detected 2 outliers using the z score method


,Salary,Z_Score,z_score
100,250000.0,5.598253,5.598253
101,300000.0,7.088169,7.088169
